In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import pickle

# Load dataset
file_path = 'smurfDS.xlsx'
df = pd.read_excel(file_path)

# Preprocessing
df['Date_Time'] = pd.to_datetime(df['Date_Time']).astype('int64') // 10**9
label_encoder = LabelEncoder()
df['Transaction_Type'] = label_encoder.fit_transform(df['Transaction_Type'])
scaler = MinMaxScaler()
df[['Withdrawal_Amt', 'Deposit_Amt']] = scaler.fit_transform(df[['Withdrawal_Amt', 'Deposit_Amt']])

# Create sequences
def create_sequences(data, sequence_length):
    sequences = []
    labels = []
    for i in range(len(data) - sequence_length):
        seq = data[i:i + sequence_length]
        label = data[i + sequence_length][1]  # withdrawal as target
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

SEQUENCE_LENGTH = 10
features = df[['Transaction_Type', 'Withdrawal_Amt', 'Deposit_Amt', 'Date_Time']].values.astype('float32')
X, y = create_sequences(features, SEQUENCE_LENGTH)
X = X.reshape((X.shape[0], X.shape[1], 4))

# Train model
model = Sequential([
    tf.keras.layers.Input(shape=(X.shape[1], X.shape[2])),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=20, batch_size=16, validation_split=0.2)

# Save model, scaler, and label_encoder using pickle
with open('train2.pickle', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler, 'label_encoder': label_encoder}, f)


Epoch 1/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.3709 - loss: 0.6197 - val_accuracy: 0.5000 - val_loss: 0.6046
Epoch 2/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3346 - loss: 0.6253 - val_accuracy: 0.5000 - val_loss: 0.6039
Epoch 3/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3783 - loss: 0.6133 - val_accuracy: 0.5000 - val_loss: 0.6027
Epoch 4/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3386 - loss: 0.6184 - val_accuracy: 0.5000 - val_loss: 0.6025
Epoch 5/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3040 - loss: 0.6407 - val_accuracy: 0.5000 - val_loss: 0.6047
Epoch 6/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3616 - loss: 0.6132 - val_accuracy: 0.5000 - val_loss: 0.6026
Epoch 7/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3364 - loss: 0.6175 - val_accuracy: 0.5000 - val_loss: 0.6025
Epoch 8/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3500 - loss: 0.6154 - val_accuracy: 0.5000 - val_loss

In [12]:
# Load pickle model and scalers
with open('train2.pickle', 'rb') as f:
    data = pickle.load(f)
model = data['model']
scaler = data['scaler']
label_encoder = data['label_encoder']

# Sample test data
test_data = pd.DataFrame({
    'Account_No': ['ACC001', 'ACC001', 'ACC001', 'ACC001', 'ACC001', 'ACC002', 'ACC002', 'ACC002'],
    'Date_Time': pd.to_datetime([
        '2025-03-18 09:00:00', '2025-03-18 09:05:00', '2025-03-18 09:10:00',
        '2025-03-18 09:15:00', '2025-03-18 09:20:00', '2025-03-18 10:00:00',
        '2025-03-18 10:10:00', '2025-03-18 10:15:00'
    ]),
    'Transaction_Type': ['Deposit', 'Withdrawal', 'Withdrawal', 'Withdrawal', 'Withdrawal', 'Deposit', 'Withdrawal', 'Withdrawal'],
    'Withdrawal_Amt': [0, 50000, 50000, 20000, 20000, 0, 15000, 10000],
    'Deposit_Amt': [250000, 0, 0, 0, 0, 50000, 0, 0]
})

# Preprocessing test data
test_data['Transaction_Type'] = label_encoder.transform(test_data['Transaction_Type'].str.strip())
test_data['Date_Time'] = test_data['Date_Time'].astype('int64') // 10**9
test_data[['Withdrawal_Amt', 'Deposit_Amt']] = scaler.transform(test_data[['Withdrawal_Amt', 'Deposit_Amt']])

# Create sequences for testing
def create_sequences(data, sequence_length):
    sequences = []
    for i in range(len(data) - sequence_length + 1):
        seq = data[i:i + sequence_length]
        sequences.append(seq)
    return np.array(sequences)

SEQUENCE_LENGTH = 5
X_test = create_sequences(test_data[['Transaction_Type', 'Withdrawal_Amt', 'Deposit_Amt', 'Date_Time']].values, SEQUENCE_LENGTH)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 4))

# Predict and show results
predictions = model.predict(X_test)
predicted_classes = (predictions > 0.30).astype(int)

for i, pred in enumerate(predicted_classes):
    status = "🔴 Smurfing Detected" if pred == 1 else "✅ Normal"
    print(f"Transaction Sequence {i + 1}: {status}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
Transaction Sequence 1: 🔴 Smurfing Detected
Transaction Sequence 2: 🔴 Smurfing Detected
Transaction Sequence 3: 🔴 Smurfing Detected
Transaction Sequence 4: 🔴 Smurfing Detected
